# Visual Question Answering using BLIP

## TA Activity - Complete Tutorial

This notebook provides a comprehensive introduction to Visual Question Answering (VQA) using the BLIP model. You will learn:

1. What is Visual Question Answering?
2. Understanding the BLIP Architecture
3. Loading and Preprocessing Images
4. Asking Questions and Getting Answers
5. Building Interactive Applications

---

## 1. Setup and Installation

First, let's install the required packages. If you're running this for the first time, uncomment the following cell.

In [ ]:
# Uncomment and run this cell if packages are not installed
# !pip install torch torchvision transformers Pillow gradio tqdm

In [ ]:
# Import required libraries
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering
import matplotlib.pyplot as plt
import os

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Understanding VQA and BLIP

### What is Visual Question Answering (VQA)?

Visual Question Answering is a task where an AI system answers natural language questions about images. It requires:
- **Visual Understanding**: Understanding what's in the image
- **Language Understanding**: Understanding the question
- **Reasoning**: Combining both to generate accurate answers

### What is BLIP?

BLIP (Bootstrapping Language-Image Pre-training) is a vision-language model developed by Salesforce. It's designed to:
- Understand images and text together
- Generate natural language descriptions
- Answer questions about images

**Key Features:**
- Pre-trained on large image-text datasets
- Handles both image understanding and generation tasks
- Available in base and large variants

## 3. Loading the BLIP Model

Let's load the BLIP model for Visual Question Answering. We'll use the base model which is efficient and accurate.

In [ ]:
# Set device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model options:
# - Salesforce/blip-vqa-base: Faster, uses less memory
# - Salesforce/blip-vqa-capfilt-large: More accurate, requires more resources

model_name = "Salesforce/blip-vqa-base"

print(f"\nLoading model: {model_name}")
print("This may take a few minutes on first run (downloading model weights)...")

# Load processor and model
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForQuestionAnswering.from_pretrained(model_name)
model.to(device)
model.eval()  # Set to evaluation mode

print("\nModel loaded successfully!")

## 4. Loading and Displaying Images

Let's define functions to load and display images.

In [ ]:
def load_image(image_path):
    """
    Load an image and convert it to RGB format.
    
    Args:
        image_path: Path to the image file
        
    Returns:
        PIL.Image: Loaded image in RGB format
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    return image

def display_image(image, title="Image"):
    """
    Display an image with matplotlib.
    
    Args:
        image: PIL Image to display
        title: Title for the plot
    """
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

print("Image functions defined!")

## 5. Asking Questions

Now let's create a function to ask questions about images.

In [ ]:
def ask_question(image, question, verbose=True):
    """
    Ask a question about an image.
    
    Args:
        image: PIL Image
        question: Question string
        verbose: Whether to print the result
        
    Returns:
        str: Answer to the question
    """
    # Process inputs
    inputs = processor(image, question, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate answer
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=50)
    
    # Decode answer
    answer = processor.decode(outputs[0], skip_special_tokens=True)
    
    if verbose:
        print(f"Question: {question}")
        print(f"Answer: {answer}")
    
    return answer

def ask_multiple_questions(image, questions):
    """
    Ask multiple questions about an image.
    
    Args:
        image: PIL Image
        questions: List of question strings
        
    Returns:
        list: List of (question, answer) tuples
    """
    results = []
    for question in questions:
        answer = ask_question(image, question, verbose=False)
        results.append((question, answer))
        print(f"Q: {question}")
        print(f"A: {answer}")
        print("-" * 40)
    return results

print("Question functions defined!")

## 6. Practical Examples

Let's try the VQA system with some examples. You can use your own images by changing the path.

In [ ]:
# Example 1: Load and display an image
# Replace with your own image path
image_path = "../images/sample.jpg"  # Change this to your image path

# Check if sample image exists, if not, create a placeholder message
if os.path.exists(image_path):
    image = load_image(image_path)
    display_image(image, "Sample Image")
else:
    print(f"Image not found at {image_path}")
    print("Please update the image_path variable with a valid image path.")
    print("\nYou can use any JPG or PNG image file.")

In [ ]:
# Example 2: Ask questions about the image
# Only run this if you have loaded an image

if 'image' in dir() and image is not None:
    # Single question
    answer = ask_question(image, "What is in the image?")

In [ ]:
# Example 3: Ask multiple questions
if 'image' in dir() and image is not None:
    questions = [
        "What is in the image?",
        "How many objects are there?",
        "What colors can you see?",
        "Describe the scene.",
        "Is there a person in the image?",
        "What is the background?"
    ]
    
    results = ask_multiple_questions(image, questions)

## 7. Try Your Own Image

Use the cells below to try VQA with your own image.

In [ ]:
# Load your own image
my_image_path = "../images/your_image.jpg"  # Change this path

if os.path.exists(my_image_path):
    my_image = load_image(my_image_path)
    display_image(my_image, "Your Image")
else:
    print(f"Please update my_image_path to point to your image.")

In [ ]:
# Ask your own questions
if 'my_image' in dir() and my_image is not None:
    your_question = "What do you see in this image?"  # Change this question
    answer = ask_question(my_image, your_question)

## 8. Understanding Model Outputs

Let's explore some details about how the model works and its limitations.

In [ ]:
# Let's see the raw model outputs
if 'image' in dir() and image is not None:
    question = "What is in the image?"
    
    # Get raw inputs
    inputs = processor(image, question, return_tensors="pt")
    print("Input tensor shapes:")
    for k, v in inputs.items():
        print(f"  {k}: {v.shape}")
    
    # Generate with different max lengths
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    print("\nAnswers with different max_length:")
    for max_len in [10, 20, 50]:
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_len)
        answer = processor.decode(outputs[0], skip_special_tokens=True)
        print(f"  max_length={max_len}: {answer}")

## 9. Tips for Better Results

1. **Ask specific questions**: "What color is the car?" works better than "Tell me about the image."

2. **Use clear images**: The model performs better on clear, well-lit images.

3. **Limit question complexity**: Simple, direct questions get better answers.

4. **Experiment with phrasing**: Try rephrasing if you don't get a good answer.

In [ ]:
# Demonstrate question phrasing impact
if 'image' in dir() and image is not None:
    variations = [
        "What is this?",
        "What object is shown?",
        "What is the main subject of this image?",
        "Describe what you see."
    ]
    
    print("Comparing different question phrasings:")
    print("=" * 50)
    for q in variations:
        answer = ask_question(image, q, verbose=False)
        print(f"Q: {q}")
        print(f"A: {answer}")
        print("-" * 50)

## 10. Summary

In this notebook, you learned:

1. **VQA Fundamentals**: What Visual Question Answering is and why it's useful
2. **BLIP Model**: How to load and use the BLIP model for VQA
3. **Image Processing**: How to load and prepare images for the model
4. **Question Answering**: How to ask questions and interpret answers
5. **Best Practices**: Tips for getting better results

### Next Steps:
- Try the Gradio demo for an interactive web interface: `python scripts/gradio_demo.py`
- Experiment with the large model for better accuracy
- Build your own applications using these foundations

---